# Task 1 - DCGAN para sprites de Pokemon

Implementacion desde cero con PyTorch de una DCGAN para generar imagenes RGB de 64x64. El notebook cubre Task 1.1 (arquitecturas), Task 1.2 (entrenamiento alternado) y Task 1.3 (visualizaciones).

## 1. Imports, configuracion y reproducibilidad

In [ ]:
from pathlib import Path
import csv
import json
import os
import random
import sys
import time

import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

Z_DIM = 100
IMG_SIZE = 64
IMG_CHANNELS = 3
FEATURES_G = 64
FEATURES_D = 64
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
BETAS = (0.5, 0.999)
NUM_EPOCHS = 50
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 4))
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'pokemon'
OUTPUT_DIR = ROOT / 'outputs' / 'task1'
GRID_DIR = OUTPUT_DIR / 'grids'
GRID_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Dispositivo: {DEVICE}')
print(f'Datos: {DATA_DIR}')

## 2. Dataset

El generador termina en `Tanh`, por lo que las imagenes reales se transforman de `[0, 255]` a `[-1, 1]`. El script de descarga ya deja los sprites en RGB y 64x64; el dataset vuelve a validar ambas condiciones.

In [ ]:
class PokemonDataset(Dataset):
    def __init__(self, root: Path):
        self.files = sorted(Path(root).glob('*.png'))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]
        with Image.open(path) as image:
            image = image.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.NEAREST)
            array = np.asarray(image, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(array).permute(2, 0, 1)
        return tensor.mul(2.0).sub(1.0)

dataset = PokemonDataset(DATA_DIR)
if len(dataset) == 0:
    dataloader = None
    print('No hay sprites. Ejecute: python scripts/download_pokemon.py')
else:
    dataloader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False
    )
    sample_batch = next(iter(dataloader))
    assert sample_batch.ndim == 4 and sample_batch.shape[1:] == (3, 64, 64)
    assert -1.0 <= sample_batch.min() <= sample_batch.max() <= 1.0
    print(f'Imagenes: {len(dataset)} | batch: {tuple(sample_batch.shape)}')

## 3. Task 1.1 - Generador y discriminador

El generador expande el ruido `z` de `(batch, 100, 1, 1)` hasta `(batch, 3, 64, 64)`. El discriminador realiza el proceso inverso y entrega una probabilidad escalar por imagen. La ultima convolucion del discriminador usa `stride=1` para transformar el mapa 4x4 en 1x1, como en la arquitectura DCGAN clasica.

In [ ]:
def weights_init(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight.data, 0.0, 0.02)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight.data, 1.0, 0.02)
        nn.init.constant_(module.bias.data, 0.0)


class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, channels=IMG_CHANNELS, features=FEATURES_G):
        super().__init__()
        self.net = nn.Sequential(
            self._block(z_dim, features * 8, 4, 1, 0),
            self._block(features * 8, features * 4, 4, 2, 1),
            self._block(features * 4, features * 2, 4, 2, 1),
            self._block(features * 2, features, 4, 2, 1),
            nn.ConvTranspose2d(features, channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    @staticmethod
    def _block(in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.ConvTranspose2d(
                in_channels, out_channels, kernel_size, stride, padding, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, noise):
        return self.net(noise)


class Discriminator(nn.Module):
    def __init__(self, channels=IMG_CHANNELS, features=FEATURES_D):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            self._block(features, features * 2),
            self._block(features * 2, features * 4),
            self._block(features * 4, features * 8),
            nn.Conv2d(features * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    @staticmethod
    def _block(in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, image):
        return self.net(image).view(-1)

In [ ]:
generator = Generator().to(DEVICE)
discriminator = Discriminator().to(DEVICE)
generator.apply(weights_init)
discriminator.apply(weights_init)

test_noise = torch.randn(4, Z_DIM, 1, 1, device=DEVICE)
test_images = torch.randn(4, 3, 64, 64, device=DEVICE)
assert generator(test_noise).shape == (4, 3, 64, 64), 'Forma del generador incorrecta'
assert discriminator(test_images).shape == (4,), 'Forma del discriminador incorrecta'

g_parameters = sum(parameter.numel() for parameter in generator.parameters())
d_parameters = sum(parameter.numel() for parameter in discriminator.parameters())
print(f'Pruebas de forma correctas | G: {g_parameters:,} params | D: {d_parameters:,} params')

## 4. Task 1.2 - Entrenamiento alternado

Para el discriminador se minimiza la BCE sobre datos reales con etiqueta 1 y falsos con etiqueta 0. `detach()` evita propagar ese gradiente hacia G. Para el generador se usa un batch falso nuevo y etiqueta 1, correspondiente al objetivo no saturante de enganar al discriminador.

In [ ]:
def save_grid(images, destination, title=None, show=False):
    images = images.detach().cpu().clamp(-1, 1).add(1).div(2)
    figure, axes = plt.subplots(4, 4, figsize=(8, 8))
    for axis, image in zip(axes.flat, images[:16]):
        axis.imshow(image.permute(1, 2, 0).numpy())
        axis.axis('off')
    if title:
        figure.suptitle(title)
    figure.tight_layout()
    figure.savefig(destination, dpi=150, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(figure)


def write_history(history, destination):
    with destination.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)


def read_history(destination):
    with destination.open('r', newline='', encoding='utf-8') as file:
        return [
            {
                'epoch': int(row['epoch']),
                'loss_G': float(row['loss_G']),
                'loss_D': float(row['loss_D']),
            }
            for row in csv.DictReader(file)
        ]


def train_dcgan(generator, discriminator, dataloader, epochs=NUM_EPOCHS, resume=True):
    criterion = nn.BCELoss()
    optimizer_g = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=BETAS)
    optimizer_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=BETAS)
    fixed_noise = torch.randn(16, Z_DIM, 1, 1, device=DEVICE)
    history = []
    state_path = OUTPUT_DIR / 'training_state.pt'
    start_epoch = 1
    completed_seconds = 0.0

    if resume and state_path.exists():
        state = torch.load(state_path, map_location=DEVICE, weights_only=False)
        generator.load_state_dict(state['generator'])
        discriminator.load_state_dict(state['discriminator'])
        optimizer_g.load_state_dict(state['optimizer_g'])
        optimizer_d.load_state_dict(state['optimizer_d'])
        fixed_noise = state['fixed_noise'].to(DEVICE)
        history = state['history']
        start_epoch = int(state['epoch']) + 1
        completed_seconds = float(state.get('elapsed_seconds', 0.0))
        print(f'Reanudando desde la epoca {start_epoch}.')

    if start_epoch > epochs:
        return history, fixed_noise

    run_started = time.perf_counter()

    for epoch in range(start_epoch, epochs + 1):
        started = time.perf_counter()
        generator.train()
        discriminator.train()
        total_g = 0.0
        total_d = 0.0
        seen = 0

        for real_images in dataloader:
            real_images = real_images.to(DEVICE)
            batch_size = real_images.size(0)
            real_labels = torch.ones(batch_size, device=DEVICE)
            fake_labels = torch.zeros(batch_size, device=DEVICE)

            # Paso de D: maximizar log D(x) + log(1 - D(G(z))).
            optimizer_d.zero_grad(set_to_none=True)
            noise = torch.randn(batch_size, Z_DIM, 1, 1, device=DEVICE)
            fake_images = generator(noise)
            loss_d_real = criterion(discriminator(real_images), real_labels)
            loss_d_fake = criterion(discriminator(fake_images.detach()), fake_labels)
            loss_d = loss_d_real + loss_d_fake
            loss_d.backward()
            optimizer_d.step()

            # Paso de G: minimizar -log D(G(z)) con un batch falso nuevo.
            optimizer_g.zero_grad(set_to_none=True)
            new_noise = torch.randn(batch_size, Z_DIM, 1, 1, device=DEVICE)
            new_fake_images = generator(new_noise)
            loss_g = criterion(discriminator(new_fake_images), real_labels)
            loss_g.backward()
            optimizer_g.step()

            total_d += loss_d.item() * batch_size
            total_g += loss_g.item() * batch_size
            seen += batch_size

        epoch_result = {
            'epoch': epoch,
            'loss_G': total_g / seen,
            'loss_D': total_d / seen,
        }
        history.append(epoch_result)
        write_history(history, OUTPUT_DIR / 'history.csv')

        generator.eval()
        with torch.no_grad():
            fixed_images = generator(fixed_noise)
        save_grid(
            fixed_images, GRID_DIR / f'epoch_{epoch:03d}.png', title=f'Epoca {epoch}'
        )
        elapsed = time.perf_counter() - started
        total_elapsed = completed_seconds + time.perf_counter() - run_started
        torch.save(
            {
                'epoch': epoch,
                'generator': generator.state_dict(),
                'discriminator': discriminator.state_dict(),
                'optimizer_g': optimizer_g.state_dict(),
                'optimizer_d': optimizer_d.state_dict(),
                'fixed_noise': fixed_noise.detach().cpu(),
                'history': history,
                'elapsed_seconds': total_elapsed,
            },
            state_path,
        )
        print(
            f'Epoca {epoch:02d}/{epochs} | D: {epoch_result["loss_D"]:.4f} | '
            f'G: {epoch_result["loss_G"]:.4f} | {elapsed:.1f}s'
        )

    torch.save(generator.state_dict(), OUTPUT_DIR / 'generator_final.pt')
    torch.save(discriminator.state_dict(), OUTPUT_DIR / 'discriminator_final.pt')
    torch.save(fixed_noise.detach().cpu(), OUTPUT_DIR / 'fixed_noise.pt')
    summary = {
        'epochs': epochs,
        'batch_size': BATCH_SIZE,
        'device': str(DEVICE),
        'elapsed_seconds': completed_seconds + time.perf_counter() - run_started,
        'final_loss_G': history[-1]['loss_G'],
        'final_loss_D': history[-1]['loss_D'],
    }
    (OUTPUT_DIR / 'training_summary.json').write_text(
        json.dumps(summary, indent=2), encoding='utf-8'
    )
    return history, fixed_noise

In [ ]:
# False carga resultados terminados; True entrena o reanuda hasta 50 epocas.
RUN_TRAINING = False
history = None
fixed_noise = None

final_files = [
    OUTPUT_DIR / 'history.csv',
    OUTPUT_DIR / 'generator_final.pt',
    OUTPUT_DIR / 'discriminator_final.pt',
    OUTPUT_DIR / 'fixed_noise.pt',
]

if RUN_TRAINING:
    if dataloader is None:
        raise RuntimeError('Primero descargue los sprites con scripts/download_pokemon.py')
    if len(dataset) != 898:
        raise RuntimeError(f'Se esperaban 898 sprites y se encontraron {len(dataset)}')
    history, fixed_noise = train_dcgan(generator, discriminator, dataloader, resume=True)
elif all(path.exists() for path in final_files):
    history = read_history(OUTPUT_DIR / 'history.csv')
    generator.load_state_dict(
        torch.load(OUTPUT_DIR / 'generator_final.pt', map_location=DEVICE, weights_only=True)
    )
    discriminator.load_state_dict(
        torch.load(OUTPUT_DIR / 'discriminator_final.pt', map_location=DEVICE, weights_only=True)
    )
    fixed_noise = torch.load(
        OUTPUT_DIR / 'fixed_noise.pt', map_location=DEVICE, weights_only=True
    )
    print(f'Resultados existentes cargados: {len(history)} epocas.')
else:
    print('Pruebas listas. Use RUN_TRAINING=True para iniciar las 50 epocas.')

## 5. Task 1.3 - Grilla final y curvas de perdida

In [ ]:
def create_final_visualizations(history, generator, fixed_noise):
    generator.eval()
    with torch.no_grad():
        final_images = generator(fixed_noise)
    save_grid(
        final_images, OUTPUT_DIR / 'final_grid.png', 'Muestras finales', show=True
    )

    epochs = np.array([row['epoch'] for row in history])
    losses_g = np.array([row['loss_G'] for row in history])
    losses_d = np.array([row['loss_D'] for row in history])
    closest_index = int(np.argmin(np.abs(losses_g - losses_d)))

    figure, axis = plt.subplots(figsize=(9, 5))
    axis.plot(epochs, losses_g, label='loss_G')
    axis.plot(epochs, losses_d, label='loss_D')
    axis.scatter(epochs[closest_index], losses_g[closest_index], color='black', zorder=3)
    axis.annotate(
        f'Epoca {epochs[closest_index]}',
        (epochs[closest_index], losses_g[closest_index]),
        xytext=(8, 10), textcoords='offset points',
    )
    axis.set(xlabel='Epoca', ylabel='BCE promedio', title='Perdidas de la DCGAN')
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / 'losses.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    return int(epochs[closest_index])

def show_grid_evolution(selected_epochs=(1, 10, 20, 30, 40, 50)):
    available = [
        (epoch, GRID_DIR / f'epoch_{epoch:03d}.png')
        for epoch in selected_epochs
        if (GRID_DIR / f'epoch_{epoch:03d}.png').exists()
    ]
    if not available:
        return
    figure, axes = plt.subplots(2, 3, figsize=(15, 10))
    for axis in axes.flat:
        axis.axis('off')
    for axis, (epoch, path) in zip(axes.flat, available):
        with Image.open(path) as image:
            axis.imshow(image)
        axis.set_title(f'Epoca {epoch}')
        axis.axis('off')
    figure.suptitle('Evolucion con el mismo vector de ruido fijo')
    figure.tight_layout()
    plt.show()
    plt.close(figure)

if history is not None:
    closest_epoch = create_final_visualizations(history, generator, fixed_noise)
    show_grid_evolution()
    print(f'Perdidas mas cercanas en la epoca {closest_epoch}.')

## 6. Analisis

> Completar despues del entrenamiento con: evolucion visual, comportamiento de ambas perdidas, epoca donde fueron mas cercanas y limitaciones observadas.